In [2]:
import requests
import sys
import re
import string
import random 
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
path = tf.keras.utils.get_file('alice.txt', 'https://www.gutenberg.org/files/19033/19033-0.txt')
text = open(path, 'rb').read().decode(encoding='utf-8')

text = text[485:54815].lower()

replaced = ["[illustration]", ",", "'", "-", "\"", "_", "...", "__", "*", ":", "ù"]

for r in replaced:
    text = text.replace(r, "")

text = text.strip()


In [4]:
uniq_chars = sorted(set(text))
vocab_size = len(uniq_chars)

print(f"Printing {len(uniq_chars)} characters: {uniq_chars}")


Printing 37 characters: ['\n', '\r', ' ', '!', '(', ')', '.', ';', '?', '[', ']', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [ ]:
char2int = {u:i for i, u in enumerate(uniq_chars)}
int2char = np.array(uniq_chars)

In [ ]:
seq_length = 100
step = 3
sentences = []
next_chars = []

for i in range(0, len(text) - seq_length, step):
    sentences.append(text[i : i + seq_length])
    next_chars.append(text[i + seq_length])

print(f"Total training sequences: {len(sentences)}")

In [ ]:
X = np.zeros((len(sentences), seq_length), dtype=np.float32)
y = np.zeros(len(sentences), dtype=np.float32)

for i, sentence in enumerate(sentences):
    for j, ch in enumerate(sentence):
        X[i, j] = char2int[ch]
    y[i] = char2int[next_chars[i]]

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 128, input_length=seq_length),
    tf.keras.layers.LSTM(256, return_sequences=True),
    tf.keras.layers.LSTM(256),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])

In [ ]:
model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("Training the Baby Gpt!")

training = model.fit(X, y, batch_size=128, epochs=75, validation_split=0.1, callbacks=[early_stop])

In [131]:
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

def generate_text(length=400, temperature=0.5):
    start_index = random.randint(0, len(text) - seq_length - 1)
    seed_text = text[start_index : start_index + seq_length]
    
    print(f"--- Seed: \"{seed_text}\"")
    print("--- Generated Text: ", end="")

    for i in range(length):
        x_pred = np.zeros((1, seq_length))
        for t, char in enumerate(seed_text):
            x_pred[0, t] = char2int[char]

        preds = model.predict(x_pred, verbose=0)[0]
        
        next_index = sample(preds, temperature)
        next_char = int2char[next_index]

        seed_text = seed_text[1:] + next_char

        sys.stdout.write(next_char)
        sys.stdout.flush()
    print()

generate_text(temperature=0.3)

--- Seed: "abbit began.get to your places! shouted the queen in a voice of thunder andpeople began running abou"
--- Generated Text: t the forlen keard and the door and whis the was now and when the goor and the dear and she rabber and whis the fore

KeyboardInterrupt: 